## Домашнее задание №4. Работа с NPV моделью

*Дата выдачи: 17.08.2026*

*Дедлайн: 24.08.2026 23:59*


### Задание 1: Написать расчет IRR
- Добавить метод для расчета IRR


### Задание 2: Исследовать NPV-модель на чувствительность 
- Меняем параметры вверх-вниз, считаем NPV для новых параметров
- Объясняем, как и почему изменился NPV


### Задание 3: добавить страховку в NPV модель
- Страховка стоит 0.5% от лимита
- Подключена у 40% клиентов в статусе CUR и у 60% в статусе DLQ
- Возможно наступление страхового случая с вероятностью 0.1%, при подключенной страховке мы должны возместить всю сумму текущей задолженности
- Нужно добавить страховку в модель и сделать выводы о том, как и почему меняется NPV

Вам нужно внести изменения в код ниже. Где – подсказывать не будем, это часть задания. Добавьте комментарий с указанием места, котороые вы поменяли.

### Класс модели

In [38]:
import numpy as np

class NPVModel:

    #добавим параметров для 3 задания
    insurance_share_limit = 0.005
    insurance_share_cur = 0.40
    insurance_share_dlq = 0.60
    insurance_event_prob = 0.001

    avg_missed_payments = 1.5
    recovery = 0.50
    dlq_penalty_amount = 500
    oper_costs = 100
    collection_costs = 600
    tax_rate = 0.25
    discounting_rate = 0.30
    eq_req = 0.125
    cost_of_funds = 0.16
    acquisition_cost = 1000

    def __init__(self):
        pass

    def model_balance_calculations(self, amount, rate, term):
        '''
        Расчет модельных баланса, выплаченных процентов, регулярного платежа
        :param amount: Сумма кредита
        :param rate: Ставка
        :param term: Срок
        :return:
        balance : np.array(101) : остаток тела долга по кредиту на каждый месяц
        interest : np.array(101) : выплата по процентам каждый месяц
        regular_payment : float : размер регулярного платежа
        '''

        regular_payment = round(amount * (rate / 12) *\
                                (1 + (rate / 12)) / (1 - ((1 + rate / 12) ** (-term))))
        # График баланса и процентов
        balance = np.zeros(101)
        interest = np.zeros(101)
        balance[0] = amount

        for i in range(1, term + 1):
            int_payment = balance[i - 1] * rate / 12
            debt_payment = regular_payment - int_payment
            balance[i] = max(0, round(balance[i - 1] - debt_payment))
            interest[i] = int_payment
        return balance, interest, regular_payment

    def distribution_calc(self, portfolio_distribution, pd=None, pa=None):
        '''
        Расчет распределения по статусам
        :param portfolio_distribution: график распределения по статусам
        :return:
        cur_dist : np.array(101) : доля клиентов в статусе CUR на каждый месяц
        dlq_dist : np.array(101) : доля клиентов в статусе DLQ на каждый месяц
        act_dist : np.array(101) : доля клиентов в статусе ACT на каждый месяц
        def_dist : np.array(101) : доля клиентов в статусе DEF на каждый месяц
        clo_dist : np.array(101) : доля клиентов в статусе CLO на каждый месяц
        '''
        
        cur_dist = portfolio_distribution['CUR']
        dlq_dist = portfolio_distribution['DLQ']
        act_dist = cur_dist + dlq_dist
        def_dist = portfolio_distribution['DEF'] 
        clo_dist = portfolio_distribution['CLO']


        return cur_dist, dlq_dist, act_dist, def_dist, clo_dist

    def cur_balance_calc(self, model_balance, cur_dist):
        '''
        Расчет модельного баланса в статусе CUR
        :param model_balance: плановый график баланса
        :param cur_dist: доля клиентов в статусе CUR на каждый месяц
        :return:
        principal_balance_cur : np.array(101) : principal balance в статусе CUR
        gross_balance_cur : np.array(101) : gross balance в статусе CUR
        '''
        principal_balance_cur = model_balance*cur_dist
        gross_balance_cur = model_balance*cur_dist
        return principal_balance_cur, gross_balance_cur

    def dlq_balance_calc(self, model_balance, regular_payment, dlq_dist):
        '''
        Расчет модельного баланса в статусе DLQ
        :param model_balance: плановый график баланса
        :param regular_payment: размер регулярного платежа
        :param dlq_dist: доля клиентов в статусе DLQ на каждый месяц
        :return:
        principal_balance_dlq : np.array(101) : principal balance в статусе DLQ
        gross_balance_dlq : np.array(101) : gross balance в статусе DLQ
        '''
        principal_balance_dlq = np.append(0, model_balance[:-1])*dlq_dist
        gross_balance_dlq = (model_balance + regular_payment*self.avg_missed_payments)*dlq_dist
        return principal_balance_dlq, gross_balance_dlq

    def act_balance_calc(self, principal_balance_cur, principal_balance_dlq, gross_balance_cur, gross_balance_dlq):
        '''
        Расчет модельного баланса в статусе ACT
        :param principal_balance_cur: principal balance в статусе CUR
        :param principal_balance_dlq: principal balance в статусе DLQ
        :param gross_balance_cur: principal balance в статусе CUR
        :param gross_balance_dlq: principal balance в статусе DLQ
        :param dlq_dist: доля клиентов в статусе DLQ на каждый месяц
        :return:
        principal_balance_act : np.array(101) : principal balance в статусе ACT
        gross_balance_act : np.array(101) : gross balance в статусе ACT
        '''
        principal_balance_act = principal_balance_cur + principal_balance_dlq
        gross_balance_act = gross_balance_cur + gross_balance_dlq
        return principal_balance_act, gross_balance_act

    def def_balance_calc(self, model_balance, regular_payment, def_dist):
        '''
        Расчет модельного баланса в статусе DEF
        :param model_balance: плановый график баланса
        :param regular_payment: размер регулярного платежа
        :param def_dist: доля клиентов в статусе DEF на каждый месяц
        :return:
        principal_balance_def : np.array(101) : principal balance в статусе DEF
        gross_balance_def : np.array(101) : gross balance в статусе DEF
        '''
        principal_balance_def = np.zeros(101)
        gross_balance_def = np.zeros(101)

        def_dist_change = def_dist[4:] - def_dist[3:-1]
        principal_balance_def[4:] = np.cumsum(model_balance[:-4] * def_dist_change)
        gross_balance_def[4:] = np.cumsum((model_balance[4:] + 4 * regular_payment) * def_dist_change)

        return principal_balance_def, gross_balance_def

    def profit_calc(self, principal_balance_act, principal_balance_def, term, rate, dlq_dist):
        '''

        :param principal_balance_act: principal balance в статусе ACT
        :param principal_balance_def: principal balance в статусе DEF
        :param term: срок кредита
        :param rate: тавка по кредиту
        :param dlq_dist: доля клиентов в статусе DLQ на каждый месяц
        :return:
        profit : np.array(101) : доход на каждый месяц
        '''
        interest_profit = principal_balance_act*rate/12
        penatly_profit = dlq_dist*self.dlq_penalty_amount

        new_def_balance = np.append(principal_balance_def[1:], 0) - principal_balance_def
        recovery_profit = new_def_balance*self.recovery

        profit = interest_profit + recovery_profit + penatly_profit
        profit[term+1:] = 0

        return profit

    def loss_calc(self, gross_balance_act, principal_balance_def, term, act_dist, dlq_dist, def_dist):
        '''

        :param gross_balance_act: gross balance в статусе ACT
        :param principal_balance_def: principal balance в статусе DEF
        :param term: срок кредита
        :param act_dist: доля клиентов в статусе ACT на каждый месяц
        :param dlq_dist: доля клиентов в статусе DLQ на каждый месяц
        :param def_dist: доля клиентов в статусе DEF на каждый месяц
        :return:
        loss : np.array(101) : лосс на каждый месяц
        '''
        oper_loss = np.append(act_dist[1:], 0)*self.oper_costs
        loan_loss =  principal_balance_def - np.append(0, principal_balance_def[:-1])
        collection_loss = (np.append(dlq_dist[1:], 0) + np.append(def_dist[1:], 0))*self.collection_costs

        prev_gross_balance_act = np.append(0, gross_balance_act[:-1])
        cost_of_funds_loss = prev_gross_balance_act*(1 - self.eq_req)*self.cost_of_funds/12

        loss = loan_loss + cost_of_funds_loss + oper_loss + collection_loss
        loss[term+1:] = 0

        return loss, loan_loss, cost_of_funds_loss, oper_loss, collection_loss

    def assets_liabilities_calc(self, gross_balance_act):
        '''

        :param gross_balance_act: gross balance в статусе ACT
        :return:
        assets : np.array(101) : активы в проекте
        eq_req_curve : np.array(101) : активы, обеспеченные капиталом
        fund_req_curve : np.array(101) : активы, обеспеченные фондами
        '''

        # assets
        assets = gross_balance_act

        #liabilities
        eq_req_curve = assets*self.eq_req
        fund_req_curve = assets*(1 - self.eq_req)

        return assets, eq_req_curve, fund_req_curve

    def niat_calc(self, profit, loss):
        '''
        Расчет NIAT
        :param profit: суммарный доход на каждый месяц
        :param loss: суммарный лосс на каждый месяц
        :return:
        nibt : np.array(101) : прибыль до налогообложения
        niat : np.array(101) : прибыль после налогообложения
        tax : np.array(101) : налог в каждом месяце
        '''

        nibt = profit - loss
        tax = nibt*self.tax_rate
        niat = nibt - tax

        return nibt, niat, tax

    def cashflow_calc(self, principal_balance_act, principal_balance_def, amount, profit,
                      cost_of_funds_loss, oper_costs, collection_costs, tax, niat, eq_req_curve, fund_req_curve):
        '''
        Расчет денежных потоков
        :param principal_balance_act: principal balance в статусе DEF
        :param principal_balance_def: principal balance в статусе DEF
        :param amount: срок кредита
        :param profit: доход на каждый месяц
        :param cost_of_funds_loss: лоссы на фондирование
        :param oper_costs: операционные расходы
        :param collection_costs: расходы на коллекшн
        :param tax: налог в каждом месяце
        :param niat: доход после налогообложения
        :param eq_req_curve: активы, обеспеченные капиталом
        :param fund_req_curve: активы, обеспеченные фондами
        :return:
        cf_to_client : np.array(101) : денежный поток к клиенту
        cf_to_shareholders : np.array(101) : денежный поток к акционерам
        cf_to_debtholders : np.array(101) : денежный поток к фондам
        cf_to_cost_and_tax : np.array(101) : денежный поток на косты и налоги
        '''
        # client
        delta_principal_balance_act = (np.append(0, principal_balance_act[:100]) - principal_balance_act)
        delta_principal_balance_def = (np.append(0, principal_balance_def[:100]) - principal_balance_def)
        repayments = delta_principal_balance_act - delta_principal_balance_def + profit
        cf_to_client = np.append(amount, -repayments[1:])

        # debtoholder
        fund_req_ch = fund_req_curve - np.append(fund_req_curve[0], fund_req_curve[:-1])
        fund_req_change = np.append(fund_req_curve[0], fund_req_ch[1:])
        cf_to_debtholders = cost_of_funds_loss - fund_req_change

        # shareholders
        eq_req_ch = eq_req_curve - np.append(eq_req_curve[0], eq_req_curve[:-1])
        cf_to_shareholders = niat - np.append(eq_req_curve[0], eq_req_ch[1:])

        # cost and tax
        cf_to_cost_and_tax = oper_costs + collection_costs + tax

        return cf_to_client, cf_to_shareholders, cf_to_debtholders, cf_to_cost_and_tax

    def insurance_calc(self, amount, cur_dist, dlq_dist, gross_balance_cur, gross_balance_dlq, term):
        '''
        Расчет денежных потоков страхового продукта
        '''
        # Доходы: комиссия от лимита с застрахованных клиентов
        insured_clients_share = cur_dist * self.insurance_share_cur + dlq_dist * self.insurance_share_dlq
        insurance_profit = insured_clients_share * (amount * self.insurance_share_limit)
        insurance_profit[term + 1:] = 0

        # Расходы: выплата задолженности при наступлении страхового случая
        gross_balance_insured = (gross_balance_cur * self.insurance_share_cur + 
                                    gross_balance_dlq * self.insurance_share_dlq)
        insurance_loss = gross_balance_insured * self.insurance_event_prob
        insurance_loss[term + 1:] = 0

        return insurance_profit, insurance_loss

    def npv_calc(self, amount, rate, term, portfolio_distribution, pd=None, pa=None, for_irr_calc=False, with_insurance=False):
        '''
        Расчет NPV
        :param amount: сумма кредита
        :param rate: ставка по кредиту
        :param term: срок кредита
        :param portfolio_distribution: график распределения по статусам
        :param pd: вероятность дефолта к 12-ому месяцу
        :param pa: вероятность полного досрочного погашения к 6-ому месяцу
        :return: npv: чистая приведенная стоимость кредита
        '''

        # Считаем балансы
        model_balance, interest, regular_payment =  self.model_balance_calculations(amount, rate, term)

        # Считаем распределение
        cur_dist, dlq_dist, act_dist, def_dist, clo_dist = self.distribution_calc(portfolio_distribution)

        # Считаем балансы
        principal_balance_cur, gross_balance_cur = self.cur_balance_calc(model_balance, cur_dist)
        principal_balance_dlq, gross_balance_dlq = self.dlq_balance_calc(model_balance, regular_payment, dlq_dist)
        principal_balance_act, gross_balance_act = self.act_balance_calc(principal_balance_cur, principal_balance_dlq, gross_balance_cur, gross_balance_dlq)
        principal_balance_def, gross_balance_def = self.def_balance_calc(model_balance, regular_payment, def_dist)

        # Считаем профиты
        profit = self.profit_calc(principal_balance_act, principal_balance_def, term, rate, dlq_dist)
        loss, loan_loss, cost_of_funds_loss, oper_loss, collection_loss = self.loss_calc(gross_balance_act, principal_balance_def,
                                                                                                                  term, act_dist, dlq_dist, def_dist)
        # Задание 3 итог, добавляем страховой профит и лосс
        if with_insurance:
            insurance_profit, insurance_loss = self.insurance_calc(
                amount, cur_dist, dlq_dist, gross_balance_cur, gross_balance_dlq, term
            )
            profit = profit + insurance_profit
            loss = loss + insurance_loss
            
        nibt, niat, tax = self.niat_calc(profit, loss)

        # Считаем активы и денежные потоки
        assets, eq_req_curve, fund_req_curve = self.assets_liabilities_calc(gross_balance_act)
        cf_to_client, cf_to_shareholders, cf_to_debtholders, cf_to_cost_and_tax =\
            self.cashflow_calc(principal_balance_act, principal_balance_def, amount, profit, cost_of_funds_loss, oper_loss, collection_loss,
                               tax, niat, eq_req_curve, fund_req_curve)

        # Считаем NPV
        disc_curve = np.array([1 / ((1 + self.discounting_rate) ** (i / 12.)) for i in range(101)])
        pv = np.round(np.sum(disc_curve * cf_to_shareholders)).astype(int)
        npv = pv - self.acquisition_cost

        if for_irr_calc:
            return npv, cf_to_shareholders

        return npv

    def irr_calc(self, amount, rate, term, portfolio_distribution, pd=None, pa=None):
        '''
        Расчет IRR
        :param full_cf: сумма кредита
        :param r: стартуем от нашей ставки дисконтирования
        :param t: срок кредита / 12
        :param amount: сумма кредита
        :param rate: ставка по кредиту
        :param term: срок кредита
        :param portfolio_distribution: график распределения по статусам
        :param pd: вероятность дефолта к 12-ому месяцу
        :param pa: вероятность полного досрочного погашения к 6-ому месяцу
        :return: npv: чистая приведенная стоимость кредита
        '''
        # Чтобы не пересчитывать заново
        _, cf_shareholders = self.npv_calc(amount, rate, term, portfolio_distribution, pd, pa, for_irr_calc=True)
        
        full_cf = cf_shareholders.copy()
        full_cf[0] -= self.acquisition_cost
        # для степеней каждый месяц
        t = np.arange(len(full_cf)) / 12.0
        r = self.discounting_rate
        
        # находить значение будем с помощью метода Ньютона–Рафсона
        for _ in range(50):
            # NPV
            npv_r = np.sum(full_cf / ((1.0 + r) ** t))
            
            # Если значение функции уже практически 0
            if abs(npv_r) < 0.001:
                return r
                
            # происводная NPV
            df_npv_r = np.sum(-t * full_cf / ((1.0 + r) ** (t + 1.0)))
            
            # Защита от деления на ноль
            if df_npv_r == 0:
                break
                
            # Шаг Ньютона
            r_next = r - npv_r / df_npv_r
            
            # Проверка малости шага
            if abs(r_next - r) < 0.000001:
                return r_next
                
            r = r_next
            
        return r

### Пример использования модели

In [39]:
cur_dist = np.zeros(101)
dlq_dist = np.zeros(101)
clo_dist = np.zeros(101)
def_dist = np.zeros(101)

dlq_dist[1:12] += 0.05

def_dist[6:] += 0.03
def_dist[9:] += 0.03

def_dist[12:] += dlq_dist[11]

clo_dist[3:] += 0.10
clo_dist[6:] += 0.10
clo_dist[12:] = 1 - def_dist[12]

cur_dist = 1 - def_dist - clo_dist - dlq_dist

portfolio_distribution = {
    'CUR' : cur_dist,
    'DLQ' : dlq_dist,
    'DEF' : def_dist,
    'CLO' : clo_dist
         }

In [3]:
import plotly.express as px
import pandas as pd

df = pd.DataFrame({
    'CUR': cur_dist,
    'DLQ': dlq_dist,
    'DEF': def_dist,
    'CLO': clo_dist
})

df = df.reset_index().rename(columns={'index': 'Номер выписки'})
df = df.melt(id_vars='Номер выписки', value_vars=['DEF', 'DLQ', 'CUR', 'CLO'], 
             var_name='Status', value_name='Доля')

fig = px.area(df, x='Номер выписки', y='Доля', color='Status', 
              title='Распределение статусов по номерам выписки',
              color_discrete_map={
                  'DEF': '#D62728',
                  'DLQ': '#FF7F0E',
                  'CUR': '#2CA02C',
                  'CLO': '#1F77B4',
              },
              category_orders={'Status': ['DEF', 'DLQ', 'CUR', 'CLO']}
             )

fig.update_layout(height=600)
fig.update_xaxes(range=[0, 15])

fig.show()

In [40]:
model = NPVModel()
model.npv_calc(amount=200000, term=12, rate=0.24, portfolio_distribution=portfolio_distribution)

np.int64(-1419)

### Проверка работы задания №1

In [45]:
model.irr_calc(amount=200000, term=12, rate=0.24, portfolio_distribution=portfolio_distribution)

np.float64(0.10718311873120008)

### Задание №2

In [42]:
npv_base = model.npv_calc(100000, 0.2, 12, portfolio_distribution)

print(f'base npv value: {npv_base}')

# Исследуем ставку дисконтирования:
model.discounting_rate = 0.2
npv_disc_down = model.npv_calc(100000, 0.2, 12, portfolio_distribution)

print(f'Exp 1: discounting_rate = 0.2  NPV: {npv_disc_down}')

model.discounting_rate = 0.4  # изменили вверх
npv_disc_up = model.npv_calc(100000, 0.2, 12, portfolio_distribution)

print(f'Exp 2: discounting_rate = 0.4  NPV: {npv_disc_up}')
model.discounting_rate = 0.30

# Исследуем стоимость фондирования:
model.cost_of_funds = 0.10  # изменили вниз
npv_cof_down = model.npv_calc(100000, 0.2, 12, portfolio_distribution)

print(f'Exp 1: cost_of_funds = 0.1     NPV: {npv_cof_down}')

model.cost_of_funds = 0.22  # изменили вверх
npv_cof_up = model.npv_calc(100000, 0.2, 12, portfolio_distribution)

print(f'Exp 2: cost_of_funds = 0.22    NPV: {npv_cof_up}')
model.cost_of_funds = 0.16

base npv value: -3026
Exp 1: discounting_rate = 0.2  NPV: -2727
Exp 2: discounting_rate = 0.4  NPV: -3292
Exp 1: cost_of_funds = 0.1     NPV: -1224
Exp 2: cost_of_funds = 0.22    NPV: -4829


Я не стал менять остальные атрибуты, потому что они относятся к операционным, а очевидно, что увелчивая расходы операционные, мы уменьшаем NPV, и уменьшая их мы увеличиваем NPV.

Для изменений выше:
discounting_rate, с увелечением этого атрибута мы уменьшаем нашу прибыль, так как discounting_rate по сути говорит нам порог прибыльности для акционеров. Их вклад не так велик поэтому абсолютное значение не так сильно изменилось.

cost_of_funds, как видим здесь такая же картина только уже фондирования бОльшую часть портфеля занимает, поэтому и вклад в NPV существенее.

Можно еще было поменять процент налогов tax_rate, но там было бы довольно предсказуемое изменение просто бы порезали прибыль с другим коэф.

### Задание №3

Добавление страховки в NPV

In [43]:
npv_without_ins = model.npv_calc(200000, 0.2, 12, portfolio_distribution, with_insurance=False)
npv_with_ins = model.npv_calc(200000, 0.2, 12, portfolio_distribution, with_insurance=True)

print(f"NPV without insurance: {npv_without_ins:.0f}")
print(f"NPV with insurance:  {npv_with_ins:.0f}")
print(f"effect insurance:  {npv_with_ins - npv_without_ins:.0f}")

NPV without insurance: -4205
NPV with insurance:  -1727
effect insurance:  2478


### Место для комментариев